In [3]:
import cv2
import os
import json
import numpy as np
from collections import defaultdict
from datetime import datetime
from ultralytics import YOLO


# paths
FRAMES_ROOT     = r"N:\Final Preparation\FULL_DATASET\new\testing\negitive\standardized_vedio\frame\frames"
LSTM_DIR        = r"N:\Final Preparation\FULL_DATASET\new\testing\negitive\standardized_vedio\frame\lstm_dataset"
POSE_OUTPUT_DIR = r"N:\Final Preparation\FULL_DATASET\new\testing\negitive\standardized_vedio\frame\pose_annotated"
MODEL_PATH      = r"N:\Final Preparation\model\yolov8m-pose.pt"

IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp')

CONFIDENCE_THRESHOLD       = 0.3
IOU_THRESHOLD              = 0.5
SPATIAL_MATCHING_THRESHOLD = 0.25
MAX_FRAMES_TO_REMEMBER     = 100
MERGE_TEMPORAL_GAP         = 50

WINDOW_SIZE         = 30
STRIDE              = 10
MIN_SEQUENCE_LENGTH = 30
KEYPOINT_DIMENSION  = 51
SAVE_KEYPOINTS      = True

COLORS = [
    (0, 255, 255), (255, 0, 255), (255, 255, 0), (0, 255, 0),
    (255, 128, 0), (128, 0, 255), (255, 128, 128), (128, 255, 128)
]
SKELETON = [
    (0,1),(0,2),(1,3),(2,4),(0,5),(0,6),(5,7),(7,9),
    (6,8),(8,10),(5,6),(5,11),(6,12),(11,12),(11,13),(13,15),(12,14),(14,16)
]

os.makedirs(LSTM_DIR,        exist_ok=True)
os.makedirs(POSE_OUTPUT_DIR, exist_ok=True)

OUTPUT_FOLDER_NAMES = {
    os.path.basename(LSTM_DIR),
    os.path.basename(POSE_OUTPUT_DIR),
}


def normalize_keypoints(keypoints, frame_w, frame_h):
    kps = keypoints.copy()
    kps[:, 0] = kps[:, 0] / frame_w
    kps[:, 1] = kps[:, 1] / frame_h
    kps[:, :2] = np.clip(kps[:, :2], 0.0, 1.0)
    return kps.flatten().astype(np.float32)


def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0]); y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2]); y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    if inter == 0:
        return 0.0
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    return inter / (area1 + area2 - inter + 1e-6)


def create_windows(sequence):
    T = len(sequence)
    windows = []
    if T < WINDOW_SIZE:
        return windows
    for start in range(0, T - WINDOW_SIZE + 1, STRIDE):
        windows.append(sequence[start : start + WINDOW_SIZE])
    last_win = sequence[T - WINDOW_SIZE : T]
    if not np.array_equal(windows[-1], last_win):
        windows.append(last_win)
    return windows


def draw_pose(frame, detections):
    annotated = frame.copy()
    h, w = annotated.shape[:2]

    for det in detections:
        pid   = det['person_id']
        color = COLORS[pid % len(COLORS)]
        kpts  = np.array(det['keypoints_xy']).reshape(17, 2)
        x1, y1, x2, y2 = map(int, det['bbox'])

        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)

        label = f"Person {pid}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
        cv2.rectangle(annotated, (x1, y1 - th - 8), (x1 + tw + 6, y1), color, -1)
        cv2.putText(annotated, label, (x1 + 3, y1 - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        for conn in SKELETON:
            pt1 = tuple(map(int, kpts[conn[0]]))
            pt2 = tuple(map(int, kpts[conn[1]]))
            if pt1 != (0, 0) and pt2 != (0, 0):
                cv2.line(annotated, pt1, pt2, color, 2)

        for x, y in kpts:
            if x > 0 and y > 0:
                cv2.circle(annotated, (int(x), int(y)), 5, (255, 255, 255), -1)
                cv2.circle(annotated, (int(x), int(y)), 3, color, -1)

    cv2.rectangle(annotated, (0, 0), (w, 32), (0, 0, 0), -1)
    cv2.putText(annotated, f"Persons detected: {len(detections)}",
                (8, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 0), 2)

    return annotated


class KeypointExtractor:

    def __init__(self, model):
        self.model = model
        self.folder_name      = None
        self.folder_path      = None
        self.frame_files      = []
        self.sequences        = {}
        self.frame_detections = {}

    def run(self, folder_path, folder_name):
        self.folder_path = folder_path
        self.folder_name = folder_name
        self.frame_files = sorted([
            f for f in os.listdir(folder_path)
            if f.lower().endswith(IMAGE_EXTENSIONS)
        ])

        if not self.frame_files:
            print(f"  no images found in {folder_name}")
            return False

        sequences        = defaultdict(list)
        person_history   = {}
        next_id          = 0
        frame_detections = {}

        for frame_idx, fname in enumerate(self.frame_files):
            frame = cv2.imread(os.path.join(folder_path, fname))
            if frame is None:
                frame_detections[fname] = []
                continue

            frame_h, frame_w = frame.shape[:2]

            results = self.model(frame, conf=CONFIDENCE_THRESHOLD,
                                 iou=IOU_THRESHOLD, verbose=False)
            result = results[0]

            if result.boxes is None or result.keypoints is None or len(result.boxes) == 0:
                frame_detections[fname] = []
                continue

            boxes    = result.boxes.xyxy.cpu().numpy()
            confs    = result.boxes.conf.cpu().numpy()
            kpts_all = result.keypoints.data.cpu().numpy()

            frame_dets = []

            for box, conf, kpts in zip(boxes, confs, kpts_all):
                best_match, best_score = None, 0.0
                for pid, history in person_history.items():
                    frames_since = frame_idx - history['last_seen']
                    if frames_since > MAX_FRAMES_TO_REMEMBER:
                        continue
                    ious    = [calculate_iou(box.tolist(), b) for b in history['bboxes'][-10:]]
                    avg_iou = np.mean(ious) if ious else 0.0
                    if   frames_since > 30: thresh = 0.15
                    elif frames_since > 10: thresh = 0.20
                    else:                   thresh = SPATIAL_MATCHING_THRESHOLD
                    if avg_iou > thresh and avg_iou > best_score:
                        best_score = avg_iou
                        best_match = pid

                person_id = best_match if best_match is not None else next_id
                if best_match is None:
                    person_history[next_id] = {'bboxes': [], 'last_seen': frame_idx}
                    next_id += 1

                person_history[person_id]['bboxes'].append(box.tolist())
                person_history[person_id]['last_seen'] = frame_idx
                if len(person_history[person_id]['bboxes']) > 10:
                    person_history[person_id]['bboxes'].pop(0)

                norm_kpts = normalize_keypoints(kpts, frame_w, frame_h)
                sequences[person_id].append({
                    'frame_idx'           : frame_idx,
                    'frame_file'          : fname,
                    'bbox'                : box.tolist(),
                    'keypoints_normalized': norm_kpts.tolist(),
                    'keypoints_xy'        : kpts[:, :2].flatten().tolist(),
                    'confidence'          : float(conf)
                })
                frame_dets.append({
                    'person_id'   : person_id,
                    'bbox'        : box.tolist(),
                    'keypoints_xy': kpts[:, :2].flatten().tolist(),
                })

            frame_detections[fname] = frame_dets

        self.sequences        = dict(sequences)
        self.frame_detections = frame_detections
        return True

    def merge_tracks(self):
        if len(self.sequences) <= 1:
            return

        track_info = []
        for tid, frames in self.sequences.items():
            frame_nums = [f['frame_idx'] for f in frames]
            track_info.append({
                'id'      : tid,
                'first'   : min(frame_nums),
                'last'    : max(frame_nums),
                'avg_bbox': np.mean([f['bbox'] for f in frames], axis=0).tolist()
            })
        track_info.sort(key=lambda x: x['first'])

        merged      = {}
        merged_into = {}

        for i, track1 in enumerate(track_info):
            if track1['id'] in merged_into:
                continue
            merged[track1['id']] = list(self.sequences[track1['id']])
            for track2 in track_info[i+1:]:
                if track2['id'] in merged_into:
                    continue
                gap = track2['first'] - track1['last']
                if not (0 < gap < MERGE_TEMPORAL_GAP):
                    continue
                if calculate_iou(track1['avg_bbox'], track2['avg_bbox']) > SPATIAL_MATCHING_THRESHOLD:
                    merged[track1['id']].extend(self.sequences[track2['id']])
                    merged_into[track2['id']] = track1['id']
                    track1['last']     = track2['last']
                    track1['avg_bbox'] = track2['avg_bbox']

        for tid in merged:
            frame_dict = {}
            for fd in merged[tid]:
                fidx = fd['frame_idx']
                if fidx not in frame_dict or fd['confidence'] > frame_dict[fidx]['confidence']:
                    frame_dict[fidx] = fd
            merged[tid] = [frame_dict[i] for i in sorted(frame_dict)]

        self.sequences = {new_id: seq for new_id, (_, seq) in enumerate(sorted(merged.items()))}

    def save_lstm(self):
        if not SAVE_KEYPOINTS:
            return 0

        total = 0
        metadata = {
            'folder_name'       : self.folder_name,
            'timestamp'         : datetime.now().isoformat(),
            'window_size'       : WINDOW_SIZE,
            'stride'            : STRIDE,
            'keypoint_dimension': KEYPOINT_DIMENSION,
            'normalization'     : 'frame_size',
            'persons'           : {}
        }

        for person_id, frames in sorted(self.sequences.items()):
            if len(frames) < MIN_SEQUENCE_LENGTH:
                continue

            sorted_frames = sorted(frames, key=lambda f: f['frame_idx'])
            sequence  = np.array([f['keypoints_normalized'] for f in sorted_frames], dtype=np.float32)
            windows   = create_windows(sequence)
            filenames = []

            for win_idx, window in enumerate(windows):
                filename = f"{self.folder_name}_Person{person_id}_win{win_idx}.npy"
                np.save(os.path.join(LSTM_DIR, filename), window)
                filenames.append(filename)
                total += 1

            metadata['persons'][str(person_id)] = {
                'num_frames'     : len(frames),
                'windows_created': len(windows),
                'frame_range'    : [sorted_frames[0]['frame_idx'], sorted_frames[-1]['frame_idx']],
                'avg_confidence' : float(np.mean([f['confidence'] for f in frames])),
                'window_files'   : filenames
            }

        with open(os.path.join(LSTM_DIR, f"{self.folder_name}_metadata.json"), 'w') as f:
            json.dump(metadata, f, indent=2)

        return total

    def save_pose_frames(self):
        out_folder = os.path.join(POSE_OUTPUT_DIR, self.folder_name)
        os.makedirs(out_folder, exist_ok=True)

        saved = 0
        for fname in self.frame_files:
            frame = cv2.imread(os.path.join(self.folder_path, fname))
            if frame is None:
                continue
            annotated = draw_pose(frame, self.frame_detections.get(fname, []))
            cv2.imwrite(os.path.join(out_folder, fname), annotated)
            saved += 1

        return saved


if not os.path.exists(FRAMES_ROOT):
    print("Frames root not found:", FRAMES_ROOT)
else:
    subfolders = sorted([
        d for d in os.listdir(FRAMES_ROOT)
        if os.path.isdir(os.path.join(FRAMES_ROOT, d)) and d not in OUTPUT_FOLDER_NAMES
    ])

    if not subfolders:
        print("No subfolders found in:", FRAMES_ROOT)
    else:
        print("Loading model...")
        model     = YOLO(MODEL_PATH)
        processor = KeypointExtractor(model)

        print("Found", len(subfolders), "folder(s), starting...")

        total_windows = total_poses = 0
        failed = []

        for idx, folder_name in enumerate(subfolders, start=1):
            folder_path = os.path.join(FRAMES_ROOT, folder_name)
            print(f"[{idx}/{len(subfolders)}] {folder_name}", end=" ... ", flush=True)

            try:
                ok = processor.run(folder_path, folder_name)
                if not ok:
                    failed.append(folder_name)
                    continue

                processor.merge_tracks()
                lstm_count = processor.save_lstm()
                pose_count = processor.save_pose_frames()

                total_windows += lstm_count
                total_poses   += pose_count
                print(f"{len(processor.sequences)} person(s) | {lstm_count} window(s) | {pose_count} pose frames")

            except Exception as e:
                failed.append(folder_name)
                print("FAILED (", e, ")")

        print("Done.", len(subfolders) - len(failed), "/", len(subfolders), "folders |",
              total_windows, "LSTM windows |", total_poses, "pose images")

        if failed:
            print("Failed:", ", ".join(failed))

Loading model...
Found 15 folder(s), starting...
[1/15] normal_001 ... 1 person(s) | 17 window(s) | 189 pose frames
[2/15] normal_002 ... 2 person(s) | 17 window(s) | 181 pose frames
[3/15] normal_003 ... 2 person(s) | 41 window(s) | 366 pose frames
[4/15] normal_004 ... 2 person(s) | 32 window(s) | 308 pose frames
[5/15] normal_006 ... 2 person(s) | 10 window(s) | 112 pose frames
[6/15] normal_007 ... 2 person(s) | 11 window(s) | 123 pose frames
[7/15] normal_008 ... 1 person(s) | 25 window(s) | 265 pose frames
[8/15] normal_009 ... 3 person(s) | 34 window(s) | 359 pose frames
[9/15] normal_010 ... 6 person(s) | 102 window(s) | 658 pose frames
[10/15] normal_121 ... 3 person(s) | 22 window(s) | 189 pose frames
[11/15] normal_123 ... 2 person(s) | 29 window(s) | 305 pose frames
[12/15] normal_124 ... 1 person(s) | 20 window(s) | 219 pose frames
[13/15] normal_125 ... 1 person(s) | 9 window(s) | 101 pose frames
[14/15] normal_126 ... 1 person(s) | 13 window(s) | 149 pose frames
[15/15] 